# Drawing 2D Shapes with the UR3e Simulator

This notebook demonstrates how to draw **any 2D shape** on a flat surface using the simulated UR3e.

The approach is simple:
1. Define a shape as a list of `(x, y)` points
2. Scale and center those points relative to the robot's current TCP
3. Convert to `TCP6D` waypoints (keeping Z and orientation constant)
4. Draw using `movel_waypoints` for smooth continuous motion

**Prerequisites:** the simulator must be running (`docker compose run --rm cpu` + launch Gazebo).

In [27]:
import sys
from pathlib import Path
from math import cos, sin, pi, radians

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from my_simulation.iscoin_sim.kinematics import (
    analytical_ik,
    forward_kinematics_matrix,
    pose_to_matrix,
    matrix_to_tcp6d,
)
from my_simulation.iscoin_sim import ISCoinSim as ISCoin
from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor



# Connect to the simulator
iscoin = ISCoin()
robot = iscoin.robot_control
print("Connected to simulator")

ISCoinSim connected to container 'iscoin_simulator'
Connected to simulator


In [28]:
 target = TCP6D.createFromMetersRadians(0.2, -0.2, 0.3, 0.0, 3.14, 0.0)
 T_desired = pose_to_matrix(target)
 print(T_desired)

[[-0.99999873  0.          0.00159265  0.2       ]
 [ 0.          1.          0.         -0.2       ]
 [-0.00159265  0.         -0.99999873  0.3       ]
 [ 0.          0.          0.          1.        ]]


In [29]:
solution = analytical_ik(T_desired)
print(solution)

[array([ 1.87388473, -1.03189735,  0.88475455,  1.71746376,  1.56927627,
       -2.83850389]), array([ 1.87388473, -0.210073  , -0.88475455,  2.66514852,  1.56927627,
       -2.83850389]), array([ 1.87388473, -1.7692268 ,  1.76247981, -1.5645247 , -1.56927627,
        0.30308877]), array([ 1.87388473, -0.16756487, -1.76247981,  0.35877299, -1.56927627,
        0.30308877]), array([-0.30382209, -2.97437694,  1.7628995 ,  2.78379348,  1.5712728 ,
        1.2669746 ]), array([-0.30382209, -1.37236379, -1.7628995 , -1.57560598,  1.5712728 ,
        1.2669746 ]), array([-0.30382209, -2.93109937,  0.88422198,  0.47760077, -1.5712728 ,
       -1.87461806]), array([-0.30382209, -2.10976431, -0.88422198,  1.42470968, -1.5712728 ,
       -1.87461806])]


In [30]:
print(solution[1])

[ 1.87388473 -0.210073   -0.88475455  2.66514852  1.56927627 -2.83850389]


In [31]:
sol = solution[0]
goto = Joint6D.createFromRadians(sol[0], sol[1], sol[2], -sol[3], -sol[4], -sol[5])
robot.movej(goto, a=radians(80), v=radians(120))

movej sent (duration=2s)
Movement OK — target reached


True

In [32]:
# Set TCP offset for the pen extending from the gripper
pen_length = 0.145  # meters
print(robot._tcp_offset)
robot.set_tcp(TCP6D.createFromMetersRadians(0, 0, pen_length, 0, 0, 0))
# print(f"TCP offset set: pen length = {pen_length*100:.0f} cm")

[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


In [33]:
# Go to home position
home = Joint6D.createFromRadians(1.1945, -1.1268, 1.0484, -1.5988, -1.5214, 1.0469)
robot.movej(home, a=radians(80), v=radians(60))

tcp_home = robot.get_actual_tcp_pose()
print(f"Home TCP: {tcp_home}")

movej sent (duration=3s)
Movement OK — target reached
Home TCP: TCP6D([-0.024210678512004084, -0.4497510194983479, 0.1620766225914286, 1.9935029354554512, 2.3179719956398306, -0.1288888311706116])


In [36]:
tcp_home = robot.get_actual_tcp_pose()
print(tcp_home)

TCP6D([-0.024210678511427497, -0.44975101949683444, 0.16207662258884364, 1.9935029354580927, 2.317971995642347, -0.12888883115741814])


In [41]:
joints = robot.get_actual_joint_positions()
T = forward_kinematics_matrix(joints.toList())
print(T[:3,3])

[-0.02521724 -0.43281554  0.30608071]


In [6]:
# --- Shape generators ---
# Each returns a list of (x, y) tuples forming a closed path.

def make_square():
    """Unit square."""
    return [(0, 0), (1, 0), (1, 1), (0, 1), (0, 0)]

def make_triangle():
    """Equilateral-ish triangle."""
    return [(0.5, 1), (0, 0), (1, 0), (0.5, 1)]

def make_circle(n=32):
    """Circle approximated with n segments."""
    return [(cos(2 * pi * i / n), sin(2 * pi * i / n)) for i in range(n + 1)]

def make_star(points=5, inner_ratio=0.4):
    """Star with alternating outer/inner vertices."""
    pts = []
    for i in range(points):
        # Outer vertex
        angle_out = 2 * pi * i / points - pi / 2
        pts.append((cos(angle_out), sin(angle_out)))
        # Inner vertex
        angle_in = angle_out + pi / points
        pts.append((inner_ratio * cos(angle_in), inner_ratio * sin(angle_in)))
    pts.append(pts[0])  # close the path
    return pts

print(f"Square:   {len(make_square())} points")
print(f"Triangle: {len(make_triangle())} points")
print(f"Circle:   {len(make_circle())} points")
print(f"Star:     {len(make_star())} points")

Square:   5 points
Triangle: 4 points
Circle:   33 points
Star:     11 points


In [7]:
def draw_shape(robot, points_2d, center_tcp, size=0.10, v=0.1):
    """Draw a 2D shape on a flat plane using the robot.

    Args:
        robot: iscoin.robot_control instance
        points_2d: list of (x, y) tuples defining the shape
        center_tcp: TCP6D pose at the center of the drawing
        size: drawing size in meters (default 10 cm)
        v: drawing velocity in m/s

    Returns:
        True if drawing succeeded, False otherwise.
    """
    if len(points_2d) < 2:
        print("Need at least 2 points")
        return False

    # Center the points around their centroid
    cx = sum(p[0] for p in points_2d) / len(points_2d)
    cy = sum(p[1] for p in points_2d) / len(points_2d)
    centered = [(p[0] - cx, p[1] - cy) for p in points_2d]

    # Normalize to [-1, 1] range
    max_extent = max(max(abs(p[0]), abs(p[1])) for p in centered)
    if max_extent == 0:
        print("All points are identical")
        return False
    normalized = [(p[0] / max_extent, p[1] / max_extent) for p in centered]

    # Scale and convert to TCP6D waypoints
    half = size / 2
    waypoints = []
    for i, (nx, ny) in enumerate(normalized):
        tcp = TCP6D.createFromMetersRadians(
            center_tcp.x + nx * half,
            center_tcp.y + ny * half,
            center_tcp.z,
            center_tcp.rx,
            center_tcp.ry,
            center_tcp.rz,
        )
        # Last waypoint: no blend radius (clean stop)
        r = 0.0 if i == len(normalized) - 1 else 0.005
        waypoints.append(TCP6DDescriptor(tcp, v=v, r=r))

    print(f"Drawing {len(waypoints)} waypoints (size={size*100:.0f} cm)...")
    result = robot.movel_waypoints(waypoints)
    print("Done!" if result else "Drawing failed (IK error?)")
    return result

## How it works

The `draw_shape()` function does three things:

1. **Centers and normalizes** the 2D points so any shape definition (regardless of its original coordinate range) fits in a `[-1, 1]` box around the centroid.
2. **Scales** by `size / 2` and **offsets** from the robot's current TCP position, mapping 2D `(x, y)` to the robot's XY plane while keeping Z and orientation constant.
3. **Converts** each point to a `TCP6DDescriptor` waypoint and sends them all to `movel_waypoints()`, which builds a single joint trajectory for smooth, continuous motion.

Adding a new shape is just writing a function that returns `[(x, y), ...]` — no robot code needed.

In [8]:
# --- Draw a square ---
print("=== Drawing a SQUARE ===")
tcp = robot.get_actual_tcp_pose()
draw_shape(robot, make_square(), tcp, size=0.10)

=== Drawing a SQUARE ===
Drawing 5 waypoints (size=10 cm)...
movel_waypoints sent (5 points, total=10s)
Movement OK — target reached
Done!


True

In [8]:
# --- Draw a triangle ---
# Return home first so each shape starts from the same position
robot.movej(home, a=radians(80), v=radians(60))

print("=== Drawing a TRIANGLE ===")
tcp = robot.get_actual_tcp_pose()
draw_shape(robot, make_triangle(), tcp, size=0.10)

movej sent (duration=3s)
Movement OK — target reached
=== Drawing a TRIANGLE ===
Drawing 4 waypoints (size=10 cm)...
movel_waypoints sent (4 points, total=8s)
Movement OK — target reached
Done!


True

In [9]:
# --- Draw a circle ---
robot.movej(home, a=radians(80), v=radians(60))

print("=== Drawing a CIRCLE ===")
tcp = robot.get_actual_tcp_pose()
draw_shape(robot, make_circle(n=32), tcp, size=0.10)

movej sent (duration=3s)
Movement OK — target reached
=== Drawing a CIRCLE ===
Drawing 33 waypoints (size=10 cm)...
movel_waypoints sent (33 points, total=66s)
Movement OK — target reached
Done!


True

In [9]:
# --- Draw a star ---
robot.movej(home, a=radians(80), v=radians(60))

print("=== Drawing a STAR ===")
tcp = robot.get_actual_tcp_pose()
draw_shape(robot, make_star(points=5), tcp, size=0.12)

movej sent (duration=3s)
Movement OK — target reached
=== Drawing a STAR ===
Drawing 11 waypoints (size=12 cm)...
movel_waypoints sent (11 points, total=22s)
Movement OK — target reached
Done!


True

In [34]:

from time import sleep

# Capture reference pose
# robot_arm.freedrive_mode()
# input("Move robot so the stylus is in the hole, then press ENTER.")
tcp_ref = robot.get_actual_tcp_pose()
# robot_arm.end_freedrive_mode()

# Move back to reference pose once
robot.movel(tcp_ref, a=1.2, v=0.25)

ROT = 0.8   # about ~28 degrees

# Helper: apply rotation around TCP
def rotate_around_tcp(tcp, rx, ry, rz):
    vals = tcp.toList()
    return TCP6D.createFromMetersRadians(
        vals[0], vals[1], vals[2],
        vals[3] + rx, vals[4] + ry, vals[5] + rz,
    )

# Perform validation rotations
axes = [
    (ROT, 0, 0),
    (-ROT, 0, 0),
    (0, ROT, 0),
    (0, -ROT, 0),
    (0, 0, ROT),
    (0, 0, -ROT)
]

for rx, ry, rz in axes:
    test_pose = rotate_around_tcp(tcp_ref, rx, ry, rz)
    print(f"Rotating: rx={rx}, ry={ry}, rz={rz}")

    robot.movel(test_pose, a=1.2, v=0.25)
    # robot.waitRobotIdleOrStopFlag()
    sleep(2)
    robot.movel(tcp_ref, a=1.2, v=0.25)
    # robot.waitRobotIdleOrStopFlag()
    sleep(1)

print("Validation complete.")


movej sent (duration=2s)
Movement OK — target reached
Rotating: rx=0.8, ry=0, rz=0
INFO: Analytical IK returned 0 valid solutions for target at distance=0.4787m (max reach ~0.457m)
ERROR: movel failed — could not find inverse kinematics solution
movej sent (duration=2s)
Movement OK — target reached
Rotating: rx=-0.8, ry=0, rz=0
movej sent (duration=2s)
Movement OK — target reached
movej sent (duration=2s)
Movement OK — target reached
Rotating: rx=0, ry=0.8, rz=0
INFO: Analytical IK returned 0 valid solutions for target at distance=0.4787m (max reach ~0.457m)
ERROR: movel failed — could not find inverse kinematics solution
movej sent (duration=2s)
Movement OK — target reached
Rotating: rx=0, ry=-0.8, rz=0
movej sent (duration=2s)
Movement OK — target reached
movej sent (duration=2s)
Movement OK — target reached
Rotating: rx=0, ry=0, rz=0.8
INFO: Analytical IK returned 0 valid solutions for target at distance=0.4787m (max reach ~0.457m)
ERROR: movel failed — could not find inverse kinema